# Lab Day 2 — Đào Gia Bảo · 2A202602793

Notebook triển khai từ khung của repo. Chọn cấu hình bằng macro-F1 val; test chỉ chạy sau khi chốt cấu hình và đủ ba seed. Báo cáo được sinh bằng tiếng Việt từ log thật.

Thứ tự: tải dữ liệu → EDA/kiểm tra → backbone → ablation → suy luận/độ trễ → chốt cấu hình → ba seed → test → workbook/báo cáo.


## 0. Cài đặt môi trường

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval.py").is_file()),
                Path("K4-Track4-Day2-Deeplearning-Advance").resolve())
if not (REPO_DIR / "eval.py").is_file():
    subprocess.run(["git", "clone", "https://github.com/kevindao94work/K4-Track4-Day2-Deeplearning-Advance.git", str(REPO_DIR)], check=True)
CODE_DIR = REPO_DIR / "submissions/2A202602793_dao_gia_bao/code"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE_DIR / "requirements.txt")], check=True)
# Lần tái lập mới dùng thư mục riêng, giữ sản phẩm và lịch sử của bài nộp.
os.environ["LAB_OUTPUT_DIR"] = str(REPO_DIR / "reproduction")
os.environ["LAB_DATA_DIR"] = str(REPO_DIR / "data")
os.environ["HF_HUB_CACHE"] = str(REPO_DIR / "data/hf_cache")
sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, str(REPO_DIR))
import torch, timm
print("torch", torch.__version__, "timm", timm.__version__)
print("CUDA", torch.cuda.is_available(), "MPS", torch.backends.mps.is_available())


### Tải dữ liệu nguồn chính thức
Trình tải giữ nguyên CSV fold 0 và kiểm tra MD5 `b7b30f96d466fba86016aa5a26606e0f` trước giải nén. ZIP chứa ảnh ở cấp gốc; trình tải đặt chúng vào `data/images`.


In [ ]:
from download_data import download
download(os.environ["LAB_DATA_DIR"])


In [ ]:
IMAGES_DIR = str(REPO_DIR / "data/images")
LABELS_DIR = str(REPO_DIR / "data/labels")


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
from dataset import load_split, check_split
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_info = check_split(train_df, val_df, test_df, IMAGES_DIR)
print("Thứ tự lớp:", split_info["class_names"])
print("Sai khác nhãn nguồn:", split_info["annotation_conflicts"])


In [ ]:
from validate_pipeline import main as validate_pipeline
validate_pipeline()
from IPython.display import display, Image
OUT_DIR = Path(os.environ["LAB_OUTPUT_DIR"])
for name in ["class_distribution.png", "class_samples.png", "augmentation_samples.png", "mixed_samples.png", "one_batch_overfit.png"]:
    display(Image(filename=str(OUT_DIR / "evidence" / name)))
print((OUT_DIR / "evidence/stage5.json").read_text())


## Bước 1 — So sánh năm backbone
Cùng công thức nền T00: 10 epoch, batch 32, 128×128, FP32, seed 0, fold 0. Các tag tiền huấn luyện được lưu trong log. Chỉ macro-F1 validation quyết định backbone đi tiếp; thời gian đo sơ bộ chưa thay cho benchmark chính thức.

In [ ]:
from workflow import backbones, audit_backbones
backbone_choice = backbones()
audit_backbones()
display(__import__("pandas").read_csv(OUT_DIR / "tables/Backbones.csv"))
print("Backbone được chọn:", backbone_choice["source_exp_id"])

## Bước 2 — Ablation ba trục và kết hợp
Mốc T00_screen dùng backbone được chọn. Trục A: finetune/frozen/scratch; B: basic/ColorJitter/CutMix; C: CE/label smoothing/focal. T07 kết hợp CutMix và label smoothing. Chọn bằng toàn bộ validation, một seed sàng lọc.

In [ ]:
from check_loader_equivalence import main as check_loader_equivalence
check_loader_equivalence()
from workflow import training
training_choice = training()
display(__import__("pandas").read_csv(OUT_DIR / "tables/Training.csv"))

## Bước 3 — Suy luận và độ trễ trên validation
Một checkpoint cố định: một view, hflip prob/logit, five-crop, 160/224, multi-scale, TS khớp NLL validation. Benchmark 10 warmup/50 lần đồng bộ, batch 1 và 32; không gồm đọc ảnh/resize/normalize CPU.

In [ ]:
from inference_lab import run_inference
inference_choice = run_inference()
display(__import__("pandas").read_csv(OUT_DIR / "tables/Inference.csv"))
display(__import__("pandas").read_csv(OUT_DIR / "tables/Latency.csv"))

## Bước 4 — Chốt cấu hình, ba seed, rồi test
1. Chốt lựa chọn bằng validation và xác minh dry run.
2. Huấn luyện F01 và T00 với seed 0,1,2; đánh giá validation và khớp nhiệt độ trước.
3. Sau khi đủ ba seed và các cổng kiểm tra đạt, chạy test đúng một vòng mỗi ID/seed, dùng ledger chống lặp.
4. Tính mean và sample std ddof=1 bằng dự đoán và evaluator chính thức.

In [ ]:
from freeze_final import freeze
frozen = freeze()
print("Đã chốt trên validation; seed:", frozen["seeds"])
from final_training import final_training
validation_final = final_training()
display(__import__("pandas").read_csv(OUT_DIR / "tables/FinalValidation.csv"))

In [ ]:
from test_once import export_all
# Chỉ gọi sau khi sáu lượt train/val hoàn tất. Cache complete không chạy model lại.
test_ledgers = export_all()
print((OUT_DIR / "evidence/test_passes.json").read_text())

In [ ]:
from official_results import official_results
# Utility gọi eval.py score/grade bằng subprocess với đường dẫn tuyệt đối OUT_DIR.
# Chỉ đọc CSV, không thêm lượt forward test.
official = official_results()
display(__import__("pandas").read_csv(OUT_DIR / "tables/Final.csv"))
display(__import__("pandas").read_csv(OUT_DIR / "tables/PerClass.csv"))
display(Image(filename=str(OUT_DIR / "evidence/confusion_test.png")))
display(Image(filename=str(OUT_DIR / "evidence/error_examples_test.png")))
print((OUT_DIR / "evidence/official_eval/grade.md").read_text())

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
from build_workbook import build_workbook
workbook_path = build_workbook()
display(__import__("pandas").read_csv(OUT_DIR / "tables/Summary.csv"))
from IPython.display import FileLink
display(FileLink(str(workbook_path)))
print("Báo cáo M4 đã nộp:", CODE_DIR.parent / "report.md")
print("Đợt tái lập mới: cập nhật phân tích tiếng Việt từ số liệu/ảnh lỗi mới, không sao chép nhận xét cũ.")

## Ghi chú tái lập và phạm vi kiểm tra

Notebook chạy cùng các module CLI, không có bản train thứ hai. Mọi chất lượng/latency trong bài nộp được đo bằng CLI trên Apple M4, không tuyên bố đã chạy cả notebook trên Colab. Các cổng và sản phẩm nằm trong `OUT_DIR`; bộ hồ sơ đã nộp nằm tại `CODE_DIR.parent`.

`best.pt`, `latest.pt`, ảnh và cache không có trên Git; notebook tải dữ liệu và train lại. Nếu phiên notebook bị ngắt sau train, chạy lại các cell để resume từ checkpoint và xác minh cache. Nếu bị ngắt giữa test, ledger sẽ từ chối chạy lại tự động: cần điều tra, không lặp test để chọn kết quả.

Phần nhận xét của báo cáo M4 gắn với hash nguồn `configs/report_sources.json`. Workbook/evaluator sinh số liệu mới; khi chạy máy/đợt khác, cập nhật nhận xét theo kết quả mới. P95 là thời gian thành phần GPU, không gồm camera và tiền xử lý CPU.
